# 14 · Caixa de água pura — valores de referência do modelo de água

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** Quais são a densidade, a compressibilidade, a estrutura (g_OO), a difusão e a viscosidade do MEU modelo de água, nas MINHAS condições de simulação, e que valores levo para os outros notebooks?

As propriedades de um modelo de água dependem do modelo **e** dos parâmetros não ligados
(cutoffs, PME, `DispCorr`), da temperatura e do barostato. Por isso a referência para as análises
do solvente no sistema com proteína é uma caixa de água pura simulada **nas mesmas condições** da
produção, e não o valor tabelado do artigo do modelo.

| Grandeza | Como é obtida | Vai para |
|---|---|---|
| densidade mássica ρ | média do termo `Density` do `.edr` | 02 (comparação) |
| densidade numérica ρ$_n$ = N/⟨V⟩ | N oxigênios de água / volume médio | 07, `DENSIDADE_REFERENCIA_NM3` |
| compressibilidade κ$_T$ = ⟨δV²⟩/(k$_B$T⟨V⟩) | flutuação do volume (NPT) | 02, `KAPPA_AGUA_EXP_BAR` |
| g$_{OO}$(r), 1º pico e nº de coordenação | histograma de pares O–O | 07 (validação, sem proteína) |
| viscosidade η | perturbação periódica, termo `1/Viscosity` do `.edr` (Hess, *J. Chem. Phys.* **2002**, 116, 209) | 11, `VISCOSIDADE_MPA_S` |
| difusão D$_\infty$ | MSD + correção de Yeh & Hummer (*J. Phys. Chem. B* **2004**, 108, 15873) | 11, `D_MODELO_M2_S` |

**Simulações esperadas** (pasta `PASTA_BASE/repN/`): a produção NPT (`md_agua.tpr`, `.gro`,
`.edr` e `md_agua_mol.xtc`, com `-pbc mol`) e, opcionalmente, uma ou mais corridas com
`cos-acceleration` (cada uma com o seu `.edr`) para a viscosidade. Caixa **cúbica**: a constante
ξ = 2,837297 da correção de Yeh–Hummer vale para caixa cúbica.

> A viscosidade por perturbação periódica depende da amplitude da aceleração: rode **duas**
> amplitudes (ex.: 0,05 e 0,02 nm ps⁻²). Se concordarem, a resposta é linear. O método de
> equilíbrio recomendado por Maginn *et al.* (*LiveCoMS* **2019**, 1, 6324) é o de Green–Kubo;
> a perturbação periódica é a que o GROMACS oferece diretamente.

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mdtraj as md
import MDAnalysis as mda
import panedr
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
    "MDTraj": md.__version__,
    "MDAnalysis": mda.__version__,
    "panedr": panedr.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

In [ ]:
# ── Onde está a caixa de água ───────────────────────────────────────────────
PASTA_BASE = Path("../../opc_box")    # pasta que contém rep1/, rep2/, ...   <- AJUSTE
SISTEMA = "agua_OPC"                # nome da pasta de resultados
REPLICAS = [1]                      # uma ou mais caixas independentes
PREFIXO_MD = "md"              # -deffnm da produção


def pasta_replica(n):
    """Pasta da réplica n."""
    return PASTA_BASE / f"rep{n}"

ARQUIVO_TOP = f"{PREFIXO_MD}.gro"
ARQUIVO_TRAJ = f"{PREFIXO_MD}_mol.xtc"       # -pbc mol
ARQUIVO_TPR = f"{PREFIXO_MD}.tpr"            # elementos pelo nº atômico (MDAnalysis >= 2.11); None = pelo nome
ARQUIVO_EDR = f"{PREFIXO_MD}.edr"            # ou .xvg do gmx energy (Density, Volume, Temperature)
ARQUIVOS = {"topologia": ARQUIVO_TOP, "trajetória": ARQUIVO_TRAJ, "energia": ARQUIVO_EDR,
            **({"tpr": ARQUIVO_TPR} if ARQUIVO_TPR else {})}
# Viscosidade: {amplitude cos-acceleration (nm ps⁻²): arquivo de energia na pasta da réplica}; {} = não há
ARQUIVOS_VISCOSIDADE = {0.05: "visc_005.edr", 0.02: "visc_002.edr"}
DESCARTE_VISC_NS = 0.5                       # início de cada corrida de viscosidade (o perfil de velocidade se forma)
VISCOSIDADE_MANUAL_MPA_S = None              # use se a viscosidade vier de outra fonte (e deixe o dicionário vazio)

# ── Amostragem ──────────────────────────────────────────────────────────────
DESCARTE_NS = 0.0               # a NPT de equilibração já foi feita antes da produção
DT_RDF_PS = 20.0                # g(r) de frames próximos é quase igual
DT_MSD_PS = 2.0                 # o desdobramento exige frames próximos (≤ 10 ps)
JANELA_MSD_NS = 10.0            # trecho da produção usado no MSD (limita a RAM)
N_BLOCOS = 5                    # incerteza por blocos dentro de cada réplica
NIVEL_CONFIANCA = 0.95

# ── Análise ─────────────────────────────────────────────────────────────────
MODELO_AGUA = "OPC"
TEMPERATURA_K = 298.15
OXIGENIO_AGUA = "water and name O"     # o MDTraj renomeia OW -> O ao ler o .gro
MAX_REFERENCIAS_RDF = 500              # oxigênios de referência sorteados (só reduz a precisão)
SEMENTE = 20260905
LARGURA_BIN_NM = 0.002
MEMORIA_LOTE_MB = 400                  # teto do bloco de distâncias em RAM
RAIO_MAX_NM = 1.2                      # limitado a metade da menor aresta da caixa
JANELA_PICO_NM = (0.25, 0.32)          # onde procurar o 1º máximo de g_OO
JANELA_MINIMO_NM = (0.30, 0.40)        # onde procurar o 1º mínimo
TAU_MAX_PS = 1000.0
FAIXA_AJUSTE_PS = (20.0, 200.0)        # confira no gráfico log-log: inclinação ~1
XI_YEH_HUMMER = 2.837297               # caixa cúbica
K_BOLTZMANN = 1.380649e-23             # J/K
N_AVOGADRO = 6.02214076e23
MASSA_MOLAR_AGUA = 18.01528            # g/mol

# ── Água real a 25 °C (para comparação) ─────────────────────────────────────
EXPERIMENTAL = {"densidade_kg_m3": 997.05,        # CRC Handbook
                "kappa_T_bar-1": 4.52e-5,          # Kell 1975
                "viscosidade_mPa_s": 0.8900,       # CRC Handbook
                "D_m2_s": 2.299e-9,                # Mills, J. Phys. Chem. 1973, 77, 685
                "r_pico_nm": 0.280, "g_pico": 2.57}  # Skinner et al., J. Chem. Phys. 2013, 138, 074506
COORDENACAO_EXP = (4.3, 4.7)

DIR_SAIDA = Path("resultados") / SISTEMA / "14_caixa_agua"
PREFIXO = "caixa_agua"
TITULO = "14 · Caixa de água pura — valores de referência"

In [ ]:
linhas = []
for n in REPLICAS:
    for papel, nome in ARQUIVOS.items():
        caminho = pasta_replica(n) / nome
        linhas.append({"replica": f"rep{n}", "arquivo": papel, "caminho": str(caminho),
                       "existe": caminho.exists(),
                       "tamanho_MB": round(caminho.stat().st_size / 1024 ** 2, 1) if caminho.exists() else np.nan})
df_arquivos = pd.DataFrame(linhas)
display(df_arquivos)
if not df_arquivos["existe"].all():
    raise FileNotFoundError("Arquivos ausentes (veja a tabela acima). Ajuste RAIZ/SISTEMA/REPLICAS.")
visc = pd.DataFrame([{"replica": f"rep{n}", "amplitude_nm_ps2": a, "caminho": str(pasta_replica(n) / f),
                      "existe": (pasta_replica(n) / f).exists()}
                     for n in REPLICAS for a, f in ARQUIVOS_VISCOSIDADE.items()])
if len(visc):
    display(visc)
    if not visc["existe"].all():
        print("[viscosidade] arquivos ausentes: as amplitudes sem arquivo ficam de fora.")

In [ ]:
def ler_energia(caminho):
    """.edr (panedr) ou .xvg (gmx energy) -> DataFrame com a coluna Time em ps.

    O panedr devolve Time como coluna E como índice: um reset_index cego duplicaria a coluna.
    """
    caminho = str(caminho)
    if caminho.endswith(".edr"):
        dados = panedr.edr_to_df(caminho)
        if "Time" not in dados.columns:
            dados = dados.reset_index().rename(columns={"index": "Time"})
        dados = dados.reset_index(drop=True)
        return dados.loc[:, ~dados.columns.duplicated()]
    linhas = open(caminho).read().splitlines()
    legendas = [l.split('"')[1] for l in linhas if l.startswith("@ s") and '"' in l]
    matriz = np.array([[float(v) for v in l.split()] for l in linhas
                       if l.strip() and not l.startswith(("#", "@"))])
    return pd.DataFrame(matriz, columns=["Time"] + legendas[:matriz.shape[1] - 1])


def coluna(dados, termo):
    """Nome da coluna do termo: exato primeiro, depois trecho do nome."""
    if termo in dados.columns:
        return termo
    achados = [c for c in dados.columns if termo.lower() in c.lower()]
    if not achados:
        raise ValueError(f"'{termo}' não está no arquivo. Termos: {list(dados.columns)}")
    return achados[0]


MAPA_RESNAMES = {"HIE": "HIS", "HID": "HIS", "HIP": "HIS", "HSD": "HIS", "HSE": "HIS",
                 "HSP": "HIS", "CYX": "CYS", "CYM": "CYS", "LYN": "LYS", "ASH": "ASP",
                 "GLH": "GLU"}


def tpr_de(n):
    """Caminho do .tpr da réplica n (None se ARQUIVO_TPR = None)."""
    return pasta_replica(n) / ARQUIVO_TPR if ARQUIVO_TPR else None


def elementos_do_tpr(tpr, top, arquivo):
    """Elemento de cada átomo da topologia, pelo número atômico gravado no .tpr.

    O .gro pode ser o sistema inteiro ou um subconjunto na MESMA ordem (o _prot.gro).
    O casamento é resíduo a resíduo, pela sequência de nomes de átomos: o MDAnalysis
    renumera os resíduos do .tpr a partir de 1, então o nº do resíduo (Leu4...) não serve.
    """
    u = mda.Universe(str(tpr))
    if not hasattr(u.atoms, "elements"):
        raise ValueError("o .tpr não trouxe números atômicos")
    if str(arquivo).endswith(".gro"):            # nomes crus (o MDTraj renomeia OW -> O etc.)
        linhas = Path(arquivo).read_text().splitlines()[2:2 + top.n_atoms]
        nomes = [l[10:15].strip() for l in linhas]
    else:
        nomes = [a.name for a in top.atoms]
    cortes = np.flatnonzero(np.diff(u.atoms.resindices)) + 1
    residuos_tpr = list(zip(np.split(u.atoms.names, cortes), np.split(u.atoms.elements, cortes)))
    elementos, j = [], 0
    for residuo in top.residues:
        alvo = [nomes[a.index] for a in residuo.atoms]
        while j < len(residuos_tpr) and list(residuos_tpr[j][0]) != alvo:
            j += 1
        if j == len(residuos_tpr):
            raise ValueError(f"{residuo} não casa com o .tpr (o .gro não é um subconjunto ordenado dele)")
        elementos += list(residuos_tpr[j][1])
        j += 1
    return elementos


def ler_topologia(arquivo, tpr=None):
    """Topologia MDTraj de um .gro/.pdb, com os elementos certos.

    * Com `tpr` (MDAnalysis >= 2.11 lê o .tpr do GROMACS 2026): o elemento vem do número
      atômico gravado no .tpr — a mesma informação que o GROMACS usa.
    * Sem `tpr` (ou se a leitura falhar): o MDTraj deduz o elemento pelo nome do átomo, e o
      FE do heme viraria flúor (19 u em vez de 55,85 u); ele é corrigido e um aviso é impresso.
    * Resíduos com nome de protonação (HIE, CYM...) recebem o nome padrão, para que
      `protein` e `resname` funcionem.
    """
    top = md.load_topology(str(arquivo))
    for residuo in top.residues:
        residuo.name = MAPA_RESNAMES.get(residuo.name.upper(), residuo.name)
    if tpr is not None and Path(tpr).exists():
        try:
            elementos = elementos_do_tpr(tpr, top, arquivo)
            for atomo, simbolo in zip(top.atoms, elementos):
                atomo.element = md.element.get_by_symbol(simbolo.capitalize()) if simbolo else md.element.virtual
            return top
        except Exception as erro:
            print(f"[aviso] .tpr não usado ({erro}); elementos deduzidos pelo nome do átomo.")
    for atomo in top.atoms:
        if atomo.residue.name == "HEM" and atomo.name.upper() == "FE":
            atomo.element = md.element.iron
    return top


def selecionar(top, consulta, n=None):
    """top.select com erro explícito se vazio (ou se não tiver n átomos)."""
    idx = top.select(consulta)
    if len(idx) == 0:
        raise ValueError(f"Seleção vazia: '{consulta}' (no MDTraj o nº do resíduo é `resSeq`).")
    if n is not None and len(idx) != n:
        raise ValueError(f"'{consulta}' casou com {len(idx)} átomos (esperado {n}).")
    return idx


def intervalo_do_arquivo_ps(xtc, top):
    """Δt entre frames gravados no .xtc (lê só os 2 primeiros frames)."""
    for lote in md.iterload(str(xtc), top=top, chunk=2, atom_indices=[0]):
        return float(np.diff(lote.time)[0])


def estride(xtc, top, dt_alvo_ps):
    """Passo de leitura que leva o Δt do arquivo ao Δt de análise."""
    dt = intervalo_do_arquivo_ps(xtc, top)
    passo = max(1, int(round(dt_alvo_ps / dt)))
    if dt_alvo_ps < dt:
        print(f"[aviso] Δt pedido {dt_alvo_ps:g} ps é menor que o do arquivo ({dt:g} ps).")
    elif abs(passo * dt - dt_alvo_ps) > 1e-6 * dt_alvo_ps:
        print(f"[aviso] Δt {dt_alvo_ps:g} ps não é múltiplo de {dt:g} ps; usando {passo * dt:g} ps.")
    return passo, dt


def producao(t_ns):
    """Máscara dos frames de produção (depois de DESCARTE_NS)."""
    t_ns = np.asarray(t_ns, dtype=float)
    mascara = t_ns >= t_ns[0] + DESCARTE_NS
    if not mascara.any():
        raise ValueError("DESCARTE_NS é maior que a trajetória.")
    return mascara



energias, linhas = {}, []
for n in REPLICAS:
    rep = f"rep{n}"
    e = ler_energia(pasta_replica(n) / ARQUIVO_EDR)
    energias[rep] = e
    top = ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    n_agua = selecionar(top, OXIGENIO_AGUA).size
    outros = sorted({r.name for r in top.residues if not r.is_water})
    linhas.append({"replica": rep, "aguas": n_agua, "atomos": top.n_atoms, "residuos_nao_agua": ", ".join(outros) or "—",
                   "pontos_energia": len(e), "inicio_ns": e["Time"].iloc[0] / 1000, "fim_ns": e["Time"].iloc[-1] / 1000,
                   "termos": ", ".join(c for c in ("Density", "Volume", "Temperature", "Pressure") if c in e.columns)})
df_leitura = pd.DataFrame(linhas)
if (df_leitura["residuos_nao_agua"] != "—").any():
    print("[!] Há resíduos que não são água na caixa: os valores deixam de ser os da água pura.")
df_leitura

## 3 · Cálculos

**3.1** Densidade, volume e temperatura do `.edr`; ρ$_n$ e κ$_T$ por réplica, com erro por
blocos. **3.2** g$_{OO}$(r) por contagem de pares (todas as águas como espécie distribuída,
`MAX_REFERENCIAS_RDF` como referência) e n(r) = 4πρ$_n$∫g r² dr. **3.3** MSD por FFT, D$_{PBC}$ do
ajuste linear em `FAIXA_AJUSTE_PS`. **3.4** Viscosidade: η = 1/⟨1/η⟩ de cada amplitude.
**3.5** Correção de Yeh–Hummer, D$_\infty$ = D$_{PBC}$ + ξk$_B$T/(6πηL), com L = ⟨V⟩$^{1/3}$.

In [ ]:
# ── 3.1 Termodinâmica ───────────────────────────────────────────────────────
def em_blocos(funcao, *series):
    """Aplica `funcao` a N_BLOCOS fatias contíguas e devolve (valor na série inteira, SEM entre blocos)."""
    fatias = np.array_split(np.arange(len(series[0])), N_BLOCOS)
    valores = [funcao(*(np.asarray(s)[f] for s in series)) for f in fatias]
    return funcao(*series), float(np.std(valores, ddof=1) / np.sqrt(len(valores)))


kappa = lambda v, T: np.var(v, ddof=1) * 1e-27 / (K_BOLTZMANN * T.mean() * v.mean()) * 1e5   # nm³ -> m³; Pa⁻¹ -> bar⁻¹
quadros, linhas = [], []
for (rep, e), n_agua in zip(energias.items(), df_leitura["aguas"]):
    t_ns = e["Time"].to_numpy(dtype=float) / 1000.0
    fase = np.where(t_ns >= t_ns[0] + DESCARTE_NS, "producao", "equilibracao")
    for termo in ("Density", "Temperature"):
        quadros.append(pd.DataFrame({"replica": rep, "observavel": "Densidade" if termo == "Density" else "Temperatura",
                                     "tempo_ns": t_ns, "valor": e[coluna(e, termo)].to_numpy(dtype=float), "fase": fase}))
    p = fase == "producao"
    v = e[coluna(e, "Volume")].to_numpy(dtype=float)[p]
    T = e[coluna(e, "Temperature")].to_numpy(dtype=float)[p]
    rho_n, u_rho_n = em_blocos(lambda v: n_agua / v.mean(), v)
    k_T, u_k_T = em_blocos(kappa, v, T)
    linhas.append({"replica": rep, "volume_medio_nm3": v.mean(), "aresta_L_nm": v.mean() ** (1 / 3),
                   "densidade_numerica_nm-3": rho_n, "u_densidade_numerica": u_rho_n,
                   "kappa_T_bar-1": k_T, "u_kappa_T": u_k_T})
df_series = pd.concat(quadros, ignore_index=True)
df_termo = pd.DataFrame(linhas)
df_termo

In [ ]:
# ── 3.2 g_OO(r) e número de coordenação ─────────────────────────────────────
quadros_rdf, resumo_rdf = [], []
for n in REPLICAS:
    rep = f"rep{n}"
    top = ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    ox = selecionar(top, OXIGENIO_AGUA)
    xtc = pasta_replica(n) / ARQUIVO_TRAJ
    passo, _ = estride(xtc, top, DT_RDF_PS)
    traj = md.load(str(xtc), top=top, stride=passo, atom_indices=ox)
    traj = traj[np.asarray(traj.time) / 1000 >= traj.time[0] / 1000 + DESCARTE_NS]
    raio = min(RAIO_MAX_NM, 0.5 * float(traj.unitcell_lengths.min()))
    refs = np.sort(np.random.default_rng(SEMENTE).choice(ox.size, min(MAX_REFERENCIAS_RDF, ox.size), replace=False))
    pares = np.column_stack([np.repeat(refs, ox.size), np.tile(np.arange(ox.size), refs.size)])
    pares = pares[pares[:, 0] != pares[:, 1]]                       # sem o par do átomo consigo
    bordas = np.arange(0.0, raio + 1e-12, LARGURA_BIN_NM)
    centros = 0.5 * (bordas[1:] + bordas[:-1])
    g_blocos = []
    lote = max(1, int(MEMORIA_LOTE_MB * 1e6 / (4 * len(pares))))  # frames por chamada (distâncias em float32)
    for fatia in np.array_split(np.arange(traj.n_frames), N_BLOCOS):
        gs, pesos = [], []
        for i in range(0, fatia.size, lote):
            sub = fatia[i:i + lote]
            gs.append(md.compute_rdf(traj[sub], pares, r_range=(0.0, bordas[-1]), n_bins=centros.size)[1])
            pesos.append(sub.size)
        g_blocos.append(np.average(gs, axis=0, weights=pesos))
    g_blocos = np.array(g_blocos)
    g = g_blocos.mean(axis=0)
    rho_n = float(df_termo.set_index("replica").loc[rep, "densidade_numerica_nm-3"])
    n_r = 4 * np.pi * rho_n * np.cumsum(g * centros ** 2 * LARGURA_BIN_NM)
    na_janela = lambda j: (centros >= j[0]) & (centros <= j[1])
    i_p = np.flatnonzero(na_janela(JANELA_PICO_NM))[np.argmax(g[na_janela(JANELA_PICO_NM)])]
    i_m = np.flatnonzero(na_janela(JANELA_MINIMO_NM))[np.argmin(g[na_janela(JANELA_MINIMO_NM)])]
    por_bloco = lambda i, f: [f(gb, i) for gb in g_blocos]
    resumo_rdf.append({"replica": rep, "frames": traj.n_frames, "raio_max_nm": bordas[-1],
                       "r_pico_nm": centros[i_p], "g_pico": g[i_p],
                       "u_g_pico": float(np.std(g_blocos[:, i_p], ddof=1) / np.sqrt(N_BLOCOS)),
                       "r_minimo_nm": centros[i_m], "numero_coordenacao": n_r[i_m],
                       "g_cauda": float(g[centros > 0.8 * bordas[-1]].mean())})
    quadros_rdf.append(pd.DataFrame({"replica": rep, "r_nm": centros, "g": g, "n_r": n_r}))
    del traj
df_rdf = pd.concat(quadros_rdf, ignore_index=True)
df_rdf_resumo = pd.DataFrame(resumo_rdf)
df_rdf_resumo

In [ ]:
def desdobrar(traj):
    """Posições contínuas (F, N, 3): soma os deslocamentos de imagem mínima entre frames (caixa
    triclínica) e remove o deslocamento médio do grupo a cada passo."""
    x = traj.xyz.astype(np.float64)
    caixa = traj.unitcell_vectors.astype(np.float64)
    d = np.diff(x, axis=0)
    for f in range(d.shape[0]):
        s = d[f] @ np.linalg.inv(caixa[f + 1])          # coordenadas fracionárias
        d[f] = (s - np.round(s)) @ caixa[f + 1]
    d -= d.mean(axis=1, keepdims=True)                  # remove -center e deriva do centro de massa
    return np.concatenate([x[:1], x[:1] + np.cumsum(d, axis=0)])


def msd_fft(x, n_max):
    """MSD médio (n_max + 1,) de x (F, N, 3), todas as origens, por FFT."""
    F = x.shape[0]
    r2 = (x ** 2).sum(axis=2)                           # (F, N)
    tam = 1 << (2 * F - 1).bit_length()
    soma_s2 = np.zeros(F)
    for k in range(3):
        f = np.fft.rfft(x[:, :, k], n=tam, axis=0)
        soma_s2 += np.fft.irfft(f * np.conjugate(f), n=tam, axis=0)[:F].sum(axis=1)
    s2 = soma_s2 / (x.shape[1] * (F - np.arange(F)))
    q = 2 * r2.sum(axis=1)
    s1 = np.empty(F)
    acumulado = q.sum() / x.shape[1]
    for m in range(F):
        if m:
            acumulado -= (r2[m - 1].sum() + r2[F - m].sum()) / x.shape[1]
        s1[m] = acumulado / (F - m)
    return (s1 - 2 * s2)[:n_max + 1]


# ── 3.3 MSD e D_PBC ─────────────────────────────────────────────────────────
def ajustar_d(tau, msd):
    faixa = (tau >= FAIXA_AJUSTE_PS[0]) & (tau <= FAIXA_AJUSTE_PS[1])
    if faixa.sum() < 3:
        raise ValueError("Poucos pontos em FAIXA_AJUSTE_PS: aumente JANELA_MSD_NS ou reduza DT_MSD_PS.")
    return stats.linregress(tau[faixa], msd[faixa]).slope / 6.0 * 1e-6      # nm²/ps -> m²/s


quadros_msd, linhas = [], []
for n in REPLICAS:
    rep = f"rep{n}"
    top = ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    xtc = pasta_replica(n) / ARQUIVO_TRAJ
    passo, _ = estride(xtc, top, DT_MSD_PS)
    traj = md.load(str(xtc), top=top, stride=passo, atom_indices=selecionar(top, OXIGENIO_AGUA))
    t_ns = np.asarray(traj.time) / 1000.0
    inicio = t_ns[0] + DESCARTE_NS
    traj = traj[(t_ns >= inicio) & (t_ns <= inicio + JANELA_MSD_NS)]
    x = desdobrar(traj)
    dt_ps = float(np.median(np.diff(traj.time)))
    n_max = min(int(TAU_MAX_PS / dt_ps), x.shape[0] - 1)
    tau = np.arange(n_max + 1) * dt_ps
    msd = msd_fft(x, n_max)
    d_bl = []
    for fatia in np.array_split(np.arange(x.shape[0]), N_BLOCOS):
        nb = min(n_max, len(fatia) - 1)
        d_bl.append(ajustar_d(tau[:nb + 1], msd_fft(x[fatia], nb)))
    quadros_msd.append(pd.DataFrame({"replica": rep, "tau_ps": tau, "msd_nm2": msd}))
    linhas.append({"replica": rep, "D_pbc_m2_s": ajustar_d(tau, msd), "u_D_pbc": float(np.std(d_bl, ddof=1) / np.sqrt(N_BLOCOS))})
    del traj, x
df_msd = pd.concat(quadros_msd, ignore_index=True)
df_difusao = pd.DataFrame(linhas)
df_difusao

In [ ]:
# ── 3.4 Viscosidade (perturbação periódica) ─────────────────────────────────
# o GROMACS grava 1/η em m s kg⁻¹ (= 1/(Pa s)); η = 1/⟨1/η⟩
linhas = []
for n in REPLICAS:
    for a, arq in ARQUIVOS_VISCOSIDADE.items():
        caminho = pasta_replica(n) / arq
        if not caminho.exists():
            continue
        e = ler_energia(caminho)
        inv = e[coluna(e, "1/Viscosity")].to_numpy(dtype=float)
        inv = inv[e["Time"].to_numpy(dtype=float) / 1000 >= e["Time"].iloc[0] / 1000 + DESCARTE_VISC_NS]
        media, u_media = em_blocos(np.mean, inv)
        linhas.append({"replica": f"rep{n}", "amplitude_nm_ps2": a, "pontos": inv.size,
                       "viscosidade_mPa_s": 1e3 / media, "u_viscosidade": 1e3 * u_media / media ** 2})
df_visc = pd.DataFrame(linhas)
if len(df_visc):
    por_amp = df_visc.groupby("amplitude_nm_ps2")["viscosidade_mPa_s"].mean()
    if por_amp.size > 1:
        u = df_visc.groupby("amplitude_nm_ps2")["u_viscosidade"].mean()
        diferenca = abs(por_amp.iloc[0] - por_amp.iloc[-1]) / np.hypot(u.iloc[0], u.iloc[-1])
        print(f"[viscosidade] diferença entre as amplitudes = {diferenca:.1f} incertezas "
              + ("(resposta linear: ok)" if diferenca < 2 else "(> 2: use a menor amplitude ou rode outra)"))
    amplitude_usada = min(por_amp.index)                         # a mais próxima da resposta linear
    viscosidade = float(por_amp.loc[amplitude_usada])
    u_viscosidade = float(df_visc[df_visc["amplitude_nm_ps2"] == amplitude_usada]["u_viscosidade"].mean())
    origem_visc = f"perturbação periódica, A = {amplitude_usada:g} nm ps⁻²"
elif VISCOSIDADE_MANUAL_MPA_S:
    viscosidade, u_viscosidade, origem_visc = VISCOSIDADE_MANUAL_MPA_S, np.nan, "VISCOSIDADE_MANUAL_MPA_S"
else:
    viscosidade, u_viscosidade, origem_visc = np.nan, np.nan, "ausente"
    print("[!] Sem viscosidade: D_∞ sai sem a correção de Yeh–Hummer.")
print(f"η = {viscosidade:.4g} mPa s ({origem_visc})")
df_visc

# ── 3.5 Correção de Yeh–Hummer ──────────────────────────────────────────────
L_m = df_termo.set_index("replica")["aresta_L_nm"] * 1e-9
df_difusao["L_nm"] = df_difusao["replica"].map(L_m) * 1e9
df_difusao["correcao_yeh_hummer_m2_s"] = (XI_YEH_HUMMER * K_BOLTZMANN * TEMPERATURA_K
                                          / (6 * np.pi * viscosidade * 1e-3 * df_difusao["replica"].map(L_m)))
df_difusao["D_infinito_m2_s"] = df_difusao["D_pbc_m2_s"] + df_difusao["correcao_yeh_hummer_m2_s"]
display(df_visc)
df_difusao

## 4 · Estatística

Para cada grandeza: com 2 ou mais réplicas, média ± SEM **entre réplicas** (a réplica é a unidade
estatística); com uma, o SEM entre `N_BLOCOS` blocos da produção (limite inferior da incerteza).
A densidade também recebe o SEM corrigido pela ineficiência estatística da série (Chodera *et
al.*, *JCTC* **2007**, 3, 26). Ao final, a tabela **`df_para_notebooks`** traz cada valor no
parâmetro onde ele entra.

In [ ]:
def ineficiencia_estatistica(x, t_minimo=3):
    """g = 1 + 2·τ_int (em frames) — Chodera et al., JCTC 2007, 3, 26.

    ACF não enviesada (FFT), somada até o 1º C(t) <= 0 depois de t_minimo, com o
    peso (1 − t/N) aplicado UMA vez. Igual a
    pymbar.timeseries.statistical_inefficiency(fast=False).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 3 or np.allclose(x, x[0]):
        return 1.0
    d = x - x.mean()
    var = float(d @ d) / n
    tamanho = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(d, tamanho)
    acf = np.fft.irfft(f * np.conjugate(f), tamanho)[:n] / ((n - np.arange(n)) * var)
    g = 1.0
    for t in range(1, n - 1):
        if acf[t] <= 0.0 and t > t_minimo:
            break
        g += 2.0 * acf[t] * (1.0 - t / n)
    return float(max(g, 1.0))


def t_student(graus_liberdade):
    return stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=max(graus_liberdade, 1))


def resumo_replica(y, dt_ns, observavel, replica, unidade=""):
    """Média, DP, g, N_ef e SEM corrigido de UMA série (uma réplica)."""
    y = np.asarray(y, dtype=float)
    g = ineficiencia_estatistica(y)
    n_ef = max(y.size / g, 2.0)
    dp = float(np.std(y, ddof=1))
    sem = dp / np.sqrt(n_ef)
    media = float(np.mean(y))
    return {"observavel": observavel, "replica": replica, "unidade": unidade,
            "n": y.size, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(n_ef - 1) * sem,
            "ic_superior": media + t_student(n_ef - 1) * sem,
            "minimo": float(y.min()), "maximo": float(y.max()),
            "g": g, "tau_int_ns": (g - 1.0) / 2.0 * dt_ns, "n_efetivo": n_ef}


def entre_replicas(medias, observavel, unidade=""):
    """A réplica é a unidade estatística: média ± SEM das médias das réplicas."""
    medias = np.asarray(medias, dtype=float)
    k = medias.size
    media = float(medias.mean())
    dp = float(np.std(medias, ddof=1)) if k > 1 else np.nan
    sem = dp / np.sqrt(k) if k > 1 else np.nan
    return {"observavel": observavel, "replica": "entre_replicas", "unidade": unidade,
            "n": k, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(k - 1) * sem if k > 1 else np.nan,
            "ic_superior": media + t_student(k - 1) * sem if k > 1 else np.nan,
            "minimo": float(medias.min()), "maximo": float(medias.max()),
            "g": np.nan, "tau_int_ns": np.nan, "n_efetivo": float(k)}


def estatisticas_series(df_series, unidades):
    """Por réplica (só produção) + linha 'entre_replicas' quando há 2 ou mais."""
    linhas = []
    prod = df_series[df_series["fase"] == "producao"]
    for obs, bloco in prod.groupby("observavel", sort=False):
        por_rep = [resumo_replica(g["valor"], float(np.median(np.diff(g["tempo_ns"]))),
                                  obs, rep, unidades.get(obs, ""))
                   for rep, g in bloco.groupby("replica", sort=False)]
        linhas += por_rep
        if len(por_rep) > 1:
            linhas.append(entre_replicas([l["media"] for l in por_rep], obs, unidades.get(obs, "")))
    return pd.DataFrame(linhas)


def referencia(df_stats, observavel):
    """Linha 'entre_replicas' do observável (ou a da única réplica)."""
    bloco = df_stats[df_stats["observavel"] == observavel]
    entre = bloco[bloco["replica"] == "entre_replicas"]
    return (entre if len(entre) else bloco).iloc[0]


def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()

In [ ]:
df_stats = estatisticas_series(df_series, {"Densidade": "kg m⁻³", "Temperatura": "K"})


def combinar(valores, incertezas):
    """(média, SEM): entre réplicas se houver 2+, senão a incerteza por blocos da única réplica."""
    valores = np.asarray(valores, dtype=float)
    if valores.size > 1:
        return float(valores.mean()), float(valores.std(ddof=1) / np.sqrt(valores.size))
    return float(valores[0]), float(np.asarray(incertezas, dtype=float)[0])


dens = referencia(df_stats, "Densidade")
grandezas = [
    ("densidade mássica", "kg m⁻³", dens["media"], dens["sem"], EXPERIMENTAL["densidade_kg_m3"]),
    ("densidade numérica", "nm⁻³", *combinar(df_termo["densidade_numerica_nm-3"], df_termo["u_densidade_numerica"]),
     EXPERIMENTAL["densidade_kg_m3"] * N_AVOGADRO / (MASSA_MOLAR_AGUA * 1e-3) * 1e-27),
    ("κ_T", "bar⁻¹", *combinar(df_termo["kappa_T_bar-1"], df_termo["u_kappa_T"]), EXPERIMENTAL["kappa_T_bar-1"]),
    ("posição do 1º pico de g_OO", "nm", *combinar(df_rdf_resumo["r_pico_nm"], [LARGURA_BIN_NM / 2]), EXPERIMENTAL["r_pico_nm"]),
    ("altura do 1º pico de g_OO", "", *combinar(df_rdf_resumo["g_pico"], df_rdf_resumo["u_g_pico"]), EXPERIMENTAL["g_pico"]),
    ("número de coordenação", "", *combinar(df_rdf_resumo["numero_coordenacao"], [np.nan]), np.mean(COORDENACAO_EXP)),
    ("viscosidade", "mPa s", viscosidade, u_viscosidade, EXPERIMENTAL["viscosidade_mPa_s"]),
    ("D_PBC", "m² s⁻¹", *combinar(df_difusao["D_pbc_m2_s"], df_difusao["u_D_pbc"]), np.nan),
    ("D_∞ (Yeh–Hummer)", "m² s⁻¹", *combinar(df_difusao["D_infinito_m2_s"], df_difusao["u_D_pbc"]), EXPERIMENTAL["D_m2_s"]),
]
df_resultado = pd.DataFrame(grandezas, columns=["grandeza", "unidade", "modelo", "incerteza", "agua_real_25C"])
df_resultado["razao_modelo_real"] = df_resultado["modelo"] / df_resultado["agua_real_25C"]
df_resultado["valor ± incerteza"] = [formatar(m, u) for m, u in zip(df_resultado["modelo"], df_resultado["incerteza"])]

valor = df_resultado.set_index("grandeza")
df_para_notebooks = pd.DataFrame([
    {"notebook": "07_rdf", "parametro": "DENSIDADE_REFERENCIA_NM3", "valor": f"{valor.loc['densidade numérica', 'modelo']:.4f}",
     "observacao": "densidade numérica da água do modelo"},
    {"notebook": "11_difusao_agua", "parametro": "VISCOSIDADE_MPA_S", "valor": f"{viscosidade:.4f}" if np.isfinite(viscosidade) else "None",
     "observacao": origem_visc},
    {"notebook": "11_difusao_agua", "parametro": "D_MODELO_M2_S", "valor": f"{valor.loc['D_∞ (Yeh–Hummer)', 'modelo']:.4g}",
     "observacao": "D∞ da água pura do mesmo modelo"},
    {"notebook": "11_difusao_agua", "parametro": "MODELO_AGUA", "valor": f'"{MODELO_AGUA}"', "observacao": ""},
    {"notebook": "02_densidade", "parametro": "KAPPA_AGUA_EXP_BAR", "valor": f"{valor.loc['κ_T', 'modelo']:.4g}",
     "observacao": "κ_T do modelo (no lugar do experimental)"},
])
if len(REPLICAS) == 1:
    print("[1 réplica] As incertezas são entre blocos da produção (limite inferior).")
display(df_resultado[["grandeza", "unidade", "valor ± incerteza", "agua_real_25C", "razao_modelo_real"]])
print("Valores para copiar nos outros notebooks (cole as linhas no bloco 2 de cada um):")
for l in df_para_notebooks.itertuples():
    print(f"  {l.notebook:<18} {l.parametro} = {l.valor}")
df_para_notebooks

## 5 · Gráfico (matplotlib)

**Figura 1** — densidade e temperatura da produção (série + distribuição). **Figura 2** —
A: g$_{OO}$(r) e n(r) com o 1º pico de Skinner *et al.* (2013); B: MSD em log-log com a reta de
inclinação 1 (o ajuste só vale onde as curvas a acompanham); C: o modelo relativo à água real.

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Arial"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 10                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

### Figura 1 — termodinâmica da produção

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA = 6.3                  # polegadas
ALTURA_LINHA = 2.5            # polegadas por observável
ESPESSURA = 0.6
OPACIDADE = 0.85
MEDIA_MOVEL_NS = 1.0           # janela da média móvel (ns); None = só a série bruta
OPACIDADE_BRUTA = 0.25         # série bruta ao fundo quando há média móvel
MOSTRAR_HISTOGRAMA = True      # distribuição da produção à direita de cada série
BINS = 40
LETRAS = "ABCDEFGHIJ"          # "" = sem letras
ROTULOS_Y = {"Densidade": "ρ (kg m$^{-3}$)", "Temperatura": "T (K)"}
LINHAS_REF = {"Densidade": [(EXPERIMENTAL["densidade_kg_m3"], "água real 25 °C")]}            # {observável: [(valor, texto), ...]}
LIMITES_Y = {}          # {observável: (mín, máx)}

# ── Gráfico ─────────────────────────────────────────────────────────────────
observaveis = list(df_series["observavel"].unique())
fig, eixos = plt.subplots(len(observaveis), 2 if MOSTRAR_HISTOGRAMA else 1,
                          figsize=(LARGURA, ALTURA_LINHA * len(observaveis)),
                          sharey="row", squeeze=False, layout="constrained",
                          gridspec_kw={"width_ratios": [4, 1]} if MOSTRAR_HISTOGRAMA else None)
for i, obs in enumerate(observaveis):
    ax = eixos[i, 0]
    dados = df_series[df_series["observavel"] == obs]
    for k, (rep, g) in enumerate(dados.groupby("replica", sort=False)):
        cor = CORES[k % len(CORES)]
        if MEDIA_MOVEL_NS:
            janela = max(1, int(round(MEDIA_MOVEL_NS / np.median(np.diff(g["tempo_ns"])))))
            ax.plot(g["tempo_ns"], g["valor"], color=cor, lw=ESPESSURA * 0.6, alpha=OPACIDADE_BRUTA)
            ax.plot(g["tempo_ns"], g["valor"].rolling(janela, center=True, min_periods=1).mean(),
                    color=cor, lw=ESPESSURA * 1.6, alpha=OPACIDADE, label=rep)
        else:
            ax.plot(g["tempo_ns"], g["valor"], color=cor, lw=ESPESSURA, alpha=OPACIDADE, label=rep)
    t0 = dados["tempo_ns"].min()
    ax.axvspan(t0, t0 + DESCARTE_NS, color=COR_REFERENCIA, alpha=0.08, lw=0)   # descarte
    ref = referencia(df_stats, obs)
    ax.axhline(ref["media"], color=COR_MEDIA, ls="--", lw=0.8)
    for valor, texto in LINHAS_REF.get(obs, []):
        ax.axhline(valor, color=COR_REFERENCIA, ls=":", lw=0.8)
        ax.annotate(texto, (1, valor), xycoords=("axes fraction", "data"), ha="right", va="bottom",
                    color=COR_REFERENCIA, fontsize=TAMANHO_FONTE - 2,
                    bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    ax.set_ylabel(ROTULOS_Y.get(obs, obs))
    ax.set_title(LETRAS[i] if i < len(LETRAS) else "", loc="left", fontweight="bold")
    ax.set_title(rotulo_media(ref),
                 loc="right", fontsize=TAMANHO_FONTE - 1)
    if obs in LIMITES_Y:
        ax.set_ylim(LIMITES_Y[obs])
    ax.set_xlim(df_series["tempo_ns"].min(), df_series["tempo_ns"].max())
    if i < len(observaveis) - 1:
        ax.tick_params(labelbottom=False)
    if MOSTRAR_HISTOGRAMA:
        axh = eixos[i, 1]
        axh.hist(dados.loc[dados["fase"] == "producao", "valor"], bins=BINS, density=True,
                 orientation="horizontal", color=CORES[0], alpha=0.4)
        axh.axhline(ref["media"], color=COR_MEDIA, ls="--", lw=0.8)
        axh.set_xticks([])                     # escala própria em cada linha
        axh.spines[["top", "right"]].set_visible(False)
eixos[-1, 0].set_xlabel("Tempo (ns)")
if MOSTRAR_HISTOGRAMA:
    eixos[-1, 1].set_xlabel("Distribuição")
fig.legend(*eixos[0, 0].get_legend_handles_labels(), loc="outside upper center",
           ncol=len(REPLICAS), frameon=True)
fig_series = fig
plt.show()

### Figura 2 — estrutura, difusão e comparação

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 6.3, 3.25
RAIO_FIGURA_NM = 0.8
COR_N = "#bc5090"

# ── Gráfico ─────────────────────────────────────────────────────────────────
fig, (ax_g, ax_m, ax_c) = plt.subplots(1, 3, figsize=(LARGURA, ALTURA), layout="constrained",
                                       gridspec_kw={"width_ratios": [1.2, 1.1, 1]})
for i, (rep, g) in enumerate(df_rdf.groupby("replica", sort=False)):
    ax_g.plot(g["r_nm"], g["g"], color=CORES[i % len(CORES)], lw=0.9, label=rep)
ax_g.errorbar(EXPERIMENTAL["r_pico_nm"], EXPERIMENTAL["g_pico"], yerr=0.05, xerr=0.001, fmt="o", ms=3, color=COR_REFERENCIA,
              capsize=2, lw=0.8, label="Skinner 2013")
ax_g.axhline(1, color=COR_MEDIA, ls="--", lw=0.6)
ax_n = ax_g.twinx()
ax_n.plot(df_rdf.groupby("r_nm")["n_r"].mean(), color=COR_N, lw=0.9, ls="-.")
ax_n.set(ylabel="n(r)", ylim=(0, 12))
ax_n.yaxis.label.set_color(COR_N)
ax_g.set(xlabel="r (nm)", ylabel="g$_{OO}$(r)", xlim=(0.2, RAIO_FIGURA_NM))
ax_g.legend(frameon=True, fontsize=TAMANHO_FONTE - 2)

for i, (rep, g) in enumerate(df_msd.groupby("replica", sort=False)):
    g = g[g["tau_ps"] > 0]
    ax_m.loglog(g["tau_ps"], g["msd_nm2"], color=CORES[i % len(CORES)], lw=0.9)
tau = df_msd.loc[df_msd["tau_ps"] > 0, "tau_ps"].drop_duplicates().to_numpy()
meio = np.sqrt(FAIXA_AJUSTE_PS[0] * FAIXA_AJUSTE_PS[1])
ancora = float(np.interp(meio, tau, df_msd[df_msd["tau_ps"] > 0].groupby("tau_ps")["msd_nm2"].mean().reindex(tau)))
ax_m.loglog(tau, ancora * tau / meio, color=COR_MEDIA, ls="--", lw=0.8, label="inclinação 1")
ax_m.axvspan(*FAIXA_AJUSTE_PS, color="#969696", alpha=0.15, lw=0)
ax_m.set(xlabel="τ (ps)", ylabel="MSD (nm$^2$)")
ax_m.legend(frameon=True, fontsize=TAMANHO_FONTE - 2)

comparar = df_resultado.dropna(subset=["agua_real_25C"]).query("grandeza != 'densidade numérica'")
NOMES_MPL = {"densidade mássica": "ρ", "κ_T": r"$\kappa_T$", "posição do 1º pico de g_OO": r"$r$ do 1º pico de $g_{OO}$",
             "altura do 1º pico de g_OO": r"altura do 1º pico de $g_{OO}$", "número de coordenação": "nº de coordenação",
             "viscosidade": r"$\eta$", "D_∞ (Yeh–Hummer)": r"$D_\infty$ (Yeh–Hummer)"}
y = np.arange(len(comparar))
desvio = 100 * (comparar["razao_modelo_real"] - 1)
ax_c.barh(y, desvio, color=[COR_REFERENCIA if abs(d) > 5 else CORES[0] for d in desvio], alpha=0.7,
          xerr=(100 * comparar["incerteza"] / comparar["agua_real_25C"]).fillna(0), error_kw=dict(lw=0.6))
for yi, d in zip(y, desvio):
    ax_c.annotate(f"{d:+.1f} %", (d, yi), xytext=(3 if d >= 0 else -3, 0), textcoords="offset points",
                  ha="left" if d >= 0 else "right", va="center", fontsize=TAMANHO_FONTE - 2)
ax_c.axvline(0, color=COR_MEDIA, lw=0.8)
ax_c.axvspan(-5, 5, color="#969696", alpha=0.12, lw=0)
limite = max(10, 1.35 * float(np.nanmax(np.abs(desvio))))
ax_c.set(yticks=y, yticklabels=[NOMES_MPL.get(g, g) for g in comparar["grandeza"]],
         xlabel="desvio do modelo em relação à água real (%)", xlim=(-limite, limite))
ax_c.tick_params(axis="y", labelsize=TAMANHO_FONTE - 2)
for eixo, letra in zip((ax_g, ax_m, ax_c), "ABC"):
    eixo.set_title(letra, loc="left", fontweight="bold")
fig_agua = fig
plt.show()

## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_LINHA_PX = 240        # pixels por observável
LARGURA_PX = None              # None = largura da página
PASSO = 1                     # 1 ponto a cada N (deixa o HTML leve)
MEDIA_MOVEL_NS = 1.0           # janela da média móvel (ns); None = só a série bruta
MOSTRAR_HISTOGRAMA = True      # distribuição da produção à direita de cada série
BINS = 40
LINHAS_REF = {"Densidade": [(EXPERIMENTAL["densidade_kg_m3"], "água real 25 °C")]}            # {observável: [(valor, texto), ...]}
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
observaveis = list(df_series["observavel"].unique())
fig_i = make_subplots(rows=len(observaveis), cols=2 if MOSTRAR_HISTOGRAMA else 1, shared_yaxes=True,
                      column_widths=[0.8, 0.2] if MOSTRAR_HISTOGRAMA else None,
                      horizontal_spacing=0.02, vertical_spacing=0.05)
for i, obs in enumerate(observaveis, start=1):
    dados = df_series[df_series["observavel"] == obs]
    for k, (rep, g) in enumerate(dados.groupby("replica", sort=False)):
        cor = CORES[k % len(CORES)]
        if MEDIA_MOVEL_NS:
            janela = max(1, int(round(MEDIA_MOVEL_NS / np.median(np.diff(g["tempo_ns"])))))
            suave = g["valor"].rolling(janela, center=True, min_periods=1).mean().iloc[::PASSO]
            g = g.iloc[::PASSO]
            fig_i.add_trace(go.Scattergl(x=g["tempo_ns"], y=g["valor"], mode="lines", name=rep, legendgroup=rep,
                                         showlegend=False, opacity=0.3, line=dict(color=cor, width=0.8)),
                            row=i, col=1)
            fig_i.add_trace(go.Scattergl(x=g["tempo_ns"], y=suave, mode="lines", name=rep, legendgroup=rep,
                                         showlegend=(i == 1), line=dict(color=cor, width=1.8)), row=i, col=1)
        else:
            g = g.iloc[::PASSO]
            fig_i.add_trace(go.Scattergl(x=g["tempo_ns"], y=g["valor"], mode="lines", name=rep,
                                         legendgroup=rep, showlegend=(i == 1),
                                         line=dict(color=cor, width=1)), row=i, col=1)
    ref = referencia(df_stats, obs)
    t0 = dados["tempo_ns"].min()
    fig_i.add_vrect(x0=t0, x1=t0 + DESCARTE_NS, fillcolor=COR_REFERENCIA, opacity=0.08,
                    line_width=0, row=i, col=1)
    # média: traço WebGL (uma linha de forma ficaria escondida atrás das séries Scattergl)
    fig_i.add_trace(go.Scattergl(x=[t0, dados["tempo_ns"].max()], y=[ref["media"]] * 2, mode="lines",
                                 name="média", showlegend=False,
                                 line=dict(color=COR_MEDIA, dash="dash", width=1.5)), row=i, col=1)
    fig_i.add_annotation(x=t0, y=ref["media"], text=f"média ± EP: {formatar(ref['media'], ref['sem'])}",
                         showarrow=False, xanchor="left", yanchor="bottom",
                         bgcolor="rgba(255,255,255,0.7)", row=i, col=1)
    for valor, texto in LINHAS_REF.get(obs, []):                 # referências (cristal, literatura)
        fig_i.add_trace(go.Scattergl(x=[t0, dados["tempo_ns"].max()], y=[valor] * 2, mode="lines",
                                     name=texto, showlegend=False,
                                     line=dict(color=COR_REFERENCIA, dash="dot", width=1.2)), row=i, col=1)
        fig_i.add_annotation(x=dados["tempo_ns"].max(), y=valor, text=texto, showarrow=False,
                             xanchor="right", yanchor="bottom", font=dict(color=COR_REFERENCIA),
                             bgcolor="rgba(255,255,255,0.7)", row=i, col=1)
    fig_i.update_yaxes(title_text=f"{obs} ({ref['unidade']})" if ref["unidade"] else obs,
                       row=i, col=1)
    valores_prod = dados.loc[dados["fase"] == "producao", "valor"]
    if MOSTRAR_HISTOGRAMA and np.ptp(valores_prod) > 0:                   # série constante não tem distribuição
        fig_i.add_trace(go.Histogram(y=valores_prod, nbinsy=BINS,
                                     histnorm="probability density", marker=dict(color=CORES[0], line_width=0),
                                     opacity=0.5, name="produção", showlegend=False), row=i, col=2)
        fig_i.add_hline(y=ref["media"], line=dict(color=COR_MEDIA, dash="dash", width=1), row=i, col=2)
    if MOSTRAR_HISTOGRAMA:
        fig_i.update_xaxes(showticklabels=False, row=i, col=2)   # escala própria em cada linha
fig_i.update_xaxes(matches="x", col=1)                           # séries com o mesmo eixo de tempo
for i in range(1, len(observaveis)):
    fig_i.update_xaxes(showticklabels=False, row=i, col=1)       # rótulos do tempo só embaixo
fig_i.update_xaxes(title_text="Tempo (ns)", row=len(observaveis), col=1)
if MOSTRAR_HISTOGRAMA:
    fig_i.update_xaxes(title_text="Distribuição", row=len(observaveis), col=2)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_LINHA_PX * len(observaveis) + 80,
                    width=LARGURA_PX, font=dict(family=FONTE, size=TAMANHO_FONTE_PX),
                    legend_title_text="Réplica", bargap=0)
fig_interativo = fig_i
fig_i.show()

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_PX = 420
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
# legenda: só as réplicas; Skinner e a reta de inclinação 1 estão rotulados no gráfico
fig_i = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, subplot_titles=("g_OO(r)", "MSD (log-log)"))
for i, (rep, g) in enumerate(df_rdf.groupby("replica", sort=False)):
    fig_i.add_trace(go.Scatter(x=g["r_nm"], y=g["g"], mode="lines", name=rep, legendgroup=rep,
                               line=dict(color=CORES[i % len(CORES)])), row=1, col=1)
for i, (rep, g) in enumerate(df_msd.groupby("replica", sort=False)):
    g = g[g["tau_ps"] > 0]
    fig_i.add_trace(go.Scatter(x=g["tau_ps"], y=g["msd_nm2"], mode="lines", name=rep, legendgroup=rep, showlegend=False,
                               line=dict(color=CORES[i % len(CORES)])), row=1, col=2)
fig_i.add_trace(go.Scatter(x=[EXPERIMENTAL["r_pico_nm"]], y=[EXPERIMENTAL["g_pico"]], mode="markers", showlegend=False,
                           marker=dict(color=COR_REFERENCIA, size=8), error_y=dict(array=[0.05]), name="Skinner 2013"),
                row=1, col=1)
fig_i.add_annotation(x=EXPERIMENTAL["r_pico_nm"], y=EXPERIMENTAL["g_pico"], text="Skinner 2013", showarrow=False,
                     xanchor="left", xshift=8, font=dict(color=COR_REFERENCIA), row=1, col=1)
fig_i.add_trace(go.Scatter(x=tau, y=ancora * tau / meio, mode="lines", showlegend=False, name="inclinação 1",
                           line=dict(color=COR_MEDIA, dash="dash")), row=1, col=2)
fig_i.add_vrect(x0=FAIXA_AJUSTE_PS[0], x1=FAIXA_AJUSTE_PS[1], fillcolor="#969696", opacity=0.15, line_width=0, row=1, col=2)
fig_i.update_xaxes(title_text="r (nm)", range=[0.2, 0.8], row=1, col=1)
fig_i.update_yaxes(title_text="g<sub>OO</sub>(r)", row=1, col=1)
fig_i.update_xaxes(title_text="τ (ps)", type="log", row=1, col=2)
fig_i.update_yaxes(title_text="MSD (nm²)", type="log", row=1, col=2)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_PX, legend_title_text="Réplica",
                    font=dict(family=FONTE, size=TAMANHO_FONTE_PX))
fig_estrutura_difusao = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `para_notebooks` | **o que copiar**: notebook, parâmetro e valor |
| `resultado` | cada grandeza do modelo ± incerteza, a água real a 25 °C e a razão |
| `serie` | densidade e temperatura ao longo do tempo |
| `estatistica` | média, DP, SEM, g, N_ef por réplica e `entre_replicas` |
| `termodinamica` | volume, aresta, densidade numérica e κ_T por réplica (± blocos) |
| `rdf`, `rdf_resumo` | g_OO(r) e n(r); 1º pico, 1º mínimo e coordenação por réplica |
| `msd`, `difusao` | MSD(τ); D_PBC, correção de Yeh–Hummer e D_∞ por réplica |
| `viscosidade` | η por réplica e amplitude da aceleração |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"agua_termodinamica": fig_series, "agua_estrutura_difusao": fig_agua}
INTERATIVOS = {"agua_termodinamica": fig_interativo, "agua_estrutura_difusao": fig_estrutura_difusao}
TABELAS = {"para_notebooks": df_para_notebooks, "resultado": df_resultado, "serie": df_series, "estatistica": df_stats,
          "termodinamica": df_termo, "rdf": df_rdf, "rdf_resumo": df_rdf_resumo, "msd": df_msd,
          "difusao": df_difusao, "viscosidade": df_visc}
PARAMETROS = {"modelo": MODELO_AGUA, "temperatura_k": TEMPERATURA_K, "descarte_ns": DESCARTE_NS,
              "faixa_ajuste_ps": FAIXA_AJUSTE_PS, "viscosidade": origem_visc, "n_blocos": N_BLOCOS}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados